# Local Offline RAG with Ollama

## Overview
This notebook demonstrates building a **completely offline RAG ** system using **Ollama** for local LLMs and embeddings.

### 🚀 Benefits of Local RAG:
- **100% Offline**: No internet required after setup
- **Privacy First**: Your documents never leave your machine
- **No API Costs**: Free to run unlimited queries
- **Fast**: No network latency
- **Full Control**: Customize models and parameters

### 📋 Architecture:
```
PDF Documents → Load → Split → Local Embeddings (Ollama) → ChromaDB
                                                                  ↓
User Query → Retrieve Similar Chunks → Local LLM (Ollama) → Answer
```

### 🛠️ Components:
- **Document Loader**: PyPDFLoader
- **Text Splitter**: RecursiveCharacterTextSplitter
- **Embeddings**: Ollama with nomic-embed-text (or embeddinggemma)
- **Vector Store**: ChromaDB (persistent, local)
- **LLM**: Ollama with gemma3:1b
- **Chain**: LangChain Expression Language (LCEL)

---

## 1. Prerequisites & Installation

### Required Software:
1. **Ollama**: Download from https://ollama.ai
2. **Python 3.9+**: Recommended 3.11 or 3.13

### Download Ollama Models:

Run these commands in your terminal (if you haven't already):

```bash
# Embedding model (choose one or both)
ollama pull nomic-embed-text    # Recommended: 274 MB
ollama pull embeddinggemma      # Alternative: 621 MB

# LLM for generation
ollama pull gemma3:27b          
```

**Note**: already have these models downloaded! ✓

## 2. Import Required Libraries

In [6]:
# Standard library imports
import os
import sys
from pathlib import Path


# LangChain Document Loaders
from langchain_community.document_loaders import PyPDFLoader

# LangChain Text Splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ollama Integration
from langchain_ollama import OllamaEmbeddings, ChatOllama

# ChromaDB Vector Store
from langchain_chroma import Chroma

# LangChain Core Components
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✓ All imports successful!")
print("✓ Ready for local offline RAG!")
print(f"\nPython version: {sys.version}")

✓ All imports successful!
✓ Ready for local offline RAG!

Python version: 3.12.12 (main, Oct 28 2025, 11:52:25) [Clang 20.1.4 ]


## 3. Verify Ollama Installation

Let's check that Ollama is running and our models are available.

In [1]:
# Check Ollama is running and list available models
!ollama list

NAME                        ID              SIZE      MODIFIED     
gemma3:27b                  a418f5838eaf    17 GB     13 hours ago    
qwen2.5-coder:latest        dae161e27b0e    4.7 GB    12 days ago     
kimi-k2.5:cloud             6d1c3246c608    -         2 weeks ago     
llama2:latest               78e26419b446    3.8 GB    4 weeks ago     
qwen3:8b                    500a1f067a9f    5.2 GB    7 weeks ago     
mxbai-embed-large:latest    468836162de7    669 MB    5 months ago    
nomic-embed-text:latest     0a109f422b47    274 MB    5 months ago    


In [7]:
# Test Ollama connection with a simple query
print("Testing Ollama connection...\n")

try:
    test_llm = ChatOllama(model="gemma3:27b", temperature=0)
    response = test_llm.invoke("Say 'Hello! I am running locally on your machine!'")
    
    print("✓ Ollama is working!")
    print(f"Response: {response.content}")
    
except Exception as e:
    print(f"✗ Error connecting to Ollama: {e}")
    print("\nMake sure Ollama is running. Try: ollama serve")

Testing Ollama connection...

✓ Ollama is working!
Response: Hello! I am running locally on your machine! 

It's a bit strange to *say* that, as I'm a large language model – I don't really "run" or have a location in the traditional sense! But I understand you're likely using an implementation of me (like Gemma) that is processing requests on your computer. So, consider it acknowledged! 😊



## 4. Load PDF Documents

Load your PDF documents for the RAG system.

In [8]:
# ===== CONFIGURATION: Update this path to your PDF file =====
pdf_path = "pdfs/attention_paper.pdf"  # Change this to your PDF file path
# =============================================================

# Check if file exists
if not os.path.exists(pdf_path):
    print(f"⚠️  ERROR: File '{pdf_path}' not found!")
    print("Please update the pdf_path variable with your PDF file location.")
else:
    # Load the PDF
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    # Display information
    print(f"✓ Loaded {len(documents)} pages from '{pdf_path}'")
    print(f"\n--- First Page Preview ---")
    print(f"Content (first 300 chars): {documents[0].page_content[:300]}...")
    print(f"\nMetadata: {documents[0].metadata}")
    print(f"\nTotal characters: {sum(len(doc.page_content) for doc in documents):,}")

✓ Loaded 15 pages from 'pdfs/attention_paper.pdf'

--- First Page Preview ---
Content (first 300 chars): Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par...

Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'pdfs/attention_paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}

Total characters: 39,615


## 5. Split Documents into Chunks

Break documents into smaller chunks for better retrieval precision.

In [9]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,        # Characters per chunk
    chunk_overlap=128,      # Overlap to maintain context
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Split on paragraphs, then lines, etc.
)

# Split documents
chunks = text_splitter.split_documents(documents)

# Display results
avg_chunk_size = sum(len(chunk.page_content) for chunk in chunks) / len(chunks)

print(f"✓ Split {len(documents)} documents into {len(chunks)} chunks")
print(f"\nAverage chunk size: {avg_chunk_size:.0f} characters")

# Preview chunks
print(f"\n--- Chunk Examples ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} (length: {len(chunk.page_content)} chars):")
    print(f"{chunk.page_content[:200]}...")

✓ Split 15 documents into 49 chunks

Average chunk size: 873 characters

--- Chunk Examples ---

Chunk 1 (length: 987 chars):
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...

Chunk 2 (length: 944 chars):
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more pa...

Chunk 3 (length: 986 chars):
∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started
the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Tra...


## 6. EmbeddingGemma

### About EmbeddingGemma:
- **Size**: 621 MB (larger than nomic)
- **Dimensions**: 768
- **Optimized for**: Google Gemma models
- **Use case**: Better alignment with Gemma LLMs

**Uncomment the code below to use embeddinggemma instead:**

In [ ]:
# Use embeddinggemma instead
embeddings = OllamaEmbeddings(
    model="embeddinggemma:latest"
)

# Test embeddings
print("Testing embeddinggemma embeddings...\n")
sample_text = "This is a test sentence for embeddings."
sample_embedding = embeddings.embed_query(sample_text)

print(f"✓ Embeddings model: embeddinggemma")
print(f"✓ Embedding dimension: {len(sample_embedding)}")
print(f"✓ Sample embedding (first 10 values): {sample_embedding[:10]}")

Testing embeddinggemma embeddings...

✓ Embeddings model: embeddinggemma
✓ Embedding dimension: 768
✓ Sample embedding (first 10 values): [-0.13427968, -0.019904468, 0.052416507, 0.012054956, -0.017047942, 0.05186618, 0.009137437, 0.014555919, -0.0117619205, -0.068941064]


## 7. Create ChromaDB Vector Store

### Why ChromaDB?
- **Local & Persistent**: Stores vectors on disk
- **Python 3.13 Compatible**: Works with latest Python
- **Easy to Use**: Simple API
- **Open Source**: Free and fully featured

**Note**: This step may take a minute as it processes all chunks.

In [13]:
# Create ChromaDB vector store
print(f"Creating ChromaDB vector store from {len(chunks)} chunks...")
print("This may take a minute...\n")

# Set persistent directory
persist_directory = "./chroma_db"

# Create vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="local_rag_collection"
)

print(f"✓ ChromaDB vector store created successfully!")
print(f"✓ Indexed {len(chunks)} document chunks")
print(f"✓ Stored at: {persist_directory}")
print(f"\nℹ️  Vector store persisted to disk - you can reload it later!")

Creating ChromaDB vector store from 49 chunks...
This may take a minute...

✓ ChromaDB vector store created successfully!
✓ Indexed 49 document chunks
✓ Stored at: ./chroma_db

ℹ️  Vector store persisted to disk - you can reload it later!


## 8. Create Retriever and Test

The retriever finds the most relevant chunks for a given query.

In [14]:
# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",    # Use cosine similarity
    search_kwargs={"k": 4}        # Retrieve top 4 most relevant chunks
)

print("✓ Retriever configured successfully")
print(f"  - Search type: similarity")
print(f"  - Number of documents to retrieve (k): 4")

# Test retrieval
test_query = "What is the main topic of this document?"
print(f"\n--- Retriever Test ---")
print(f"Query: '{test_query}'")

retrieved_docs = retriever.invoke(test_query)

print(f"\nRetrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Content preview: {doc.page_content[:150]}...")
    print(f"  Source: Page {doc.metadata.get('page', 'N/A')}")

✓ Retriever configured successfully
  - Search type: similarity
  - Number of documents to retrieve (k): 4

--- Retriever Test ---
Query: 'What is the main topic of this document?'

Retrieved 4 documents:

Document 1:
  Content preview: our research.
†Work performed while at Google Brain.
‡Work performed while at Google Research.
31st Conference on Neural Information Processing System...
  Source: Page 0

Document 2:
  Content preview: Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
att...
  Source: Page 3

Document 3:
  Content preview: 1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly establis...
  Source: Page 1

Document 4:
  Content preview: Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
...


## 9. Build RAG Chain

Combine retrieval and generation into a single pipeline using LCEL.

In [17]:
# Define prompt template
system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer based on the context, say that you don't know. "
    "Keep the answer concise and accurate.\n\n"
    "Context: {context}\n\n"
    "Question: {question}"
)

prompt = ChatPromptTemplate.from_template(system_prompt)

# Helper function to format documents
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain using LCEL
rag_chain = (
    {
        "context": retriever | format_docs,  # Retrieve and format docs
        "question": RunnablePassthrough()      # Pass through the question
    }
    | prompt           # Format with prompt template
    | test_llm              # Generate answer with local LLM
    | StrOutputParser() # Parse output to string
)

print("✓ RAG chain created successfully using LCEL!")
print("\nRAG Pipeline Flow:")
print("  1. User provides a query")
print("  2. Retriever finds top 4 relevant chunks (local ChromaDB)")
print("  3. Chunks are formatted as context")
print("  4. Context + question formatted with prompt template")
print("  5. Local LLM (gemma3:27b) generates answer")
print("  6. Answer parsed and returned")
print("\n🔒 Everything runs locally on machine!")

✓ RAG chain created successfully using LCEL!

RAG Pipeline Flow:
  1. User provides a query
  2. Retriever finds top 4 relevant chunks (local ChromaDB)
  3. Chunks are formatted as context
  4. Context + question formatted with prompt template
  5. Local LLM (gemma3:27b) generates answer
  6. Answer parsed and returned

🔒 Everything runs locally on machine!


## 10. Test RAG Pipeline with Example Queries

In [16]:
# Example Query 1: General question
query1 = "What is the main topic or contribution of this document?"

print(f"Query: {query1}")
print("\nProcessing locally...\n")

answer = rag_chain.invoke(query1)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

# Show source documents
print("\nSOURCE DOCUMENTS USED:")
print("=" * 80)
retrieved_docs = retriever.invoke(query1)
for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Page: {doc.metadata.get('page', 'N/A')}")
    print(f"  Content: {doc.page_content[:200]}...")
    print("-" * 80)

Query: What is the main topic or contribution of this document?

Processing locally...

ANSWER:
This document details "Scaled Dot-Product Attention" and "Multi-Head Attention," presenting a new approach to sequence modeling that addresses limitations of recurrent neural networks by enabling parallelization. It's a contribution to the field of neural machine translation and sequence transduction.


SOURCE DOCUMENTS USED:

Document 1:
  Page: 0
  Content: our research.
†Work performed while at Google Brain.
‡Work performed while at Google Research.
31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
arXiv:1706.037...
--------------------------------------------------------------------------------

Document 2:
  Page: 3
  Content: Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, ...
-------------------

In [18]:
# Example Query 2: Specific information extraction
query2 = "Can you summarize the key technical contributions or innovations mentioned?"

print(f"Query: {query2}")
print("\nProcessing locally...\n")

answer = rag_chain.invoke(query2)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

Query: Can you summarize the key technical contributions or innovations mentioned?

Processing locally...

ANSWER:
Here's a summary of the key technical contributions mentioned in the context:

*   **Self-attention:** Proposed as a replacement for RNNs.
*   **Scaled dot-product attention & multi-head attention:** Innovations in the attention mechanism.
*   **Parameter-free position representation:** A method for encoding position information.
*   **Tensor2tensor:** A codebase significantly improving results and research speed.
*   **Neural GPUs:** Learning algorithms using neural networks.
*   **Structured self-attentive sentence embedding:** A method for creating sentence embeddings.







In [19]:
# Example Query 3: Your custom question
custom_query = "What specific details are mentioned about the methodology or approach?"

print(f"Query: {custom_query}")
print("\nProcessing locally...\n")

answer = rag_chain.invoke(custom_query)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

Query: What specific details are mentioned about the methodology or approach?

Processing locally...

ANSWER:
Here's a summary of the methodologies and approaches mentioned in the context:

*   **Training:** Base models were trained for 100,000 steps (12 hours), while larger models were trained for 300,000 steps (3.5 days) with a step time of 1.0 seconds.
*   **Optimizer:** Adam optimizer was used with β1 = 0.9, β2 = 0.98, and ϵ = 10−9.
*   **Learning Rate:** The learning rate was varied using a formula involving warmup steps (4000) and step number.
*   **Regularization:** Three types of regularization were employed during training.
*   **Positional Encoding:** Positional encodings were added to input embeddings to provide sequence order information.
*   **Attention Masking:** Masking was used in scaled dot-product attention to preserve the auto-regressive property.
*   **Feed-Forward Networks:** Position-wise feed-forward networks with ReLU activation were used in encoder/decoder laye

## 11. Interactive Q&A Session

Ask own questions to the RAG system!

In [20]:
# Interactive Q&A
def ask_question(question):
    """Ask a question to the RAG system."""
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}")
    
    answer = rag_chain.invoke(question)
    
    print(f"\nAnswer: {answer}")
    print(f"{'='*80}\n")
    
    return answer

# Try it out!
# Change the question below to ask anything about your document
my_question = "What are the main findings or results?"
ask_question(my_question)


Question: What are the main findings or results?

Answer: The Transformer, a sequence transduction model based entirely on attention, achieves state-of-the-art results on WMT 2014 English-to-German and English-to-French translation tasks. It can be trained faster and at a lower cost than previous models, with a BLEU score of 41.0 on the English-to-French task. The research also found that bigger models with dropout perform better, and checkpoint averaging improves results.



'The Transformer, a sequence transduction model based entirely on attention, achieves state-of-the-art results on WMT 2014 English-to-German and English-to-French translation tasks. It can be trained faster and at a lower cost than previous models, with a BLEU score of 41.0 on the English-to-French task. The research also found that bigger models with dropout perform better, and checkpoint averaging improves results.'